<h1>CANDIDATE RESUME SCREENING SYSTEM (MINI PROJECT)</h1>

In [1]:
import random
import re
import textwrap
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Dict, List, Tuple


class ResumeScreeningError(Exception):
    pass


class InvalidCandidateError(ResumeScreeningError):
    pass


class InvalidJobError(ResumeScreeningError):
    pass


def clean_skills(skills, error_cls) -> List[str]:
    if isinstance(skills, str) or not hasattr(skills, "__iter__"):
        raise error_cls("Skills must be given as a list of skill names.")
    cleaned: List[str] = []
    for skill in skills:
        if not isinstance(skill, str) or not skill.strip():
            raise error_cls(f"Invalid skill entry: {skill!r}")
        skill = skill.strip()
        if skill.lower() not in (s.lower() for s in cleaned):
            cleaned.append(skill)
    if not cleaned:
        raise error_cls("At least one skill is required.")
    return cleaned


def normalize(text: str) -> str:
    return re.sub(r"[\s.\-]", "", text).lower()


SYNONYM_GROUPS = [
    {"genai", "gen ai", "generative ai", "llm", "prompt engineering", "rag"},
    {"ml", "machine learning"},
    {"dl", "deep learning"},
]


def expand(keyword: str) -> set:
    for group in SYNONYM_GROUPS:
        if keyword in group:
            return group
    return {keyword}


class Displayable(ABC):
    @abstractmethod
    def display_details(self) -> None:
        pass

    @abstractmethod
    def get_summary(self) -> str:
        pass


class Screenable(Displayable):
    @abstractmethod
    def get_name(self) -> str:
        pass

    @abstractmethod
    def get_candidate_type(self) -> str:
        pass

    @abstractmethod
    def match_skills(self, job: "Job") -> Tuple[List[str], List[str]]:
        pass

    @abstractmethod
    def check_experience_eligibility(self, job: "Job") -> Tuple[bool, str]:
        pass

    @abstractmethod
    def check_qualification(self, job: "Job") -> bool:
        pass


class Searchable(Displayable):
    @abstractmethod
    def match_keywords(self, keywords: List[str]) -> Tuple[int, List[str]]:
        pass

    @abstractmethod
    def get_search_keywords(self) -> List[str]:
        pass

    @abstractmethod
    def get_search_lines(self, matched: List[str],
                         numbers: Dict[str, Tuple[str, int]]) -> List[str]:
        pass


class JobRole(ABC):
    title = "Role"
    category = "General"

    @abstractmethod
    def work(self) -> str:
        pass


class AIMLRole(JobRole):
    category = "AI/ML"


class GenAIRole(JobRole):
    category = "Gen AI"


class MLEngineer(AIMLRole):
    title = "ML Engineer"

    def work(self):
        return "builds and trains machine learning models"


class MLOpsEngineer(AIMLRole):
    title = "MLOps Engineer"

    def work(self):
        return "deploys and monitors models in production"


class NLPEngineer(AIMLRole):
    title = "NLP Engineer"

    def work(self):
        return "teaches machines to understand language"


class ComputerVisionEngineer(AIMLRole):
    title = "Computer Vision Engineer"

    def work(self):
        return "builds systems that understand images and video"


class DataScientist(AIMLRole):
    title = "Data Scientist"

    def work(self):
        return "analyses data and builds predictive models"


class AIResearcher(AIMLRole):
    title = "AI Researcher"

    def work(self):
        return "researches new AI techniques and models"


class GenAIEngineer(GenAIRole):
    title = "GenAI Engineer"

    def work(self):
        return "builds apps powered by generative AI"


class PromptEngineer(GenAIRole):
    title = "Prompt Engineer"

    def work(self):
        return "designs prompts to get the best from LLMs"


class LLMEngineer(GenAIRole):
    title = "LLM Engineer"

    def work(self):
        return "fine-tunes and serves large language models"


class AIAgentDeveloper(GenAIRole):
    title = "AI Agent Developer"

    def work(self):
        return "builds AI agents that plan and use tools"


class Candidate(Screenable, Searchable):
    EMAIL_PATTERN = re.compile(r"^[\w.+-]+@[\w-]+(\.[\w-]+)+$")

    def __init__(self, candidate_id: str, name: str, email: str,
                 qualification: str, skills: List[str], role: JobRole):
        self.__candidate_id = candidate_id
        self.name = name
        self.email = email
        self.qualification = qualification
        self.skills = skills
        self.role = role

    @property
    def candidate_id(self) -> str:
        return self.__candidate_id

    @property
    def name(self) -> str:
        return self.__name

    @name.setter
    def name(self, value: str):
        if not isinstance(value, str) or not value.strip():
            raise InvalidCandidateError("Candidate name cannot be empty.")
        self.__name = value.strip()

    @property
    def email(self) -> str:
        return self.__email

    @email.setter
    def email(self, value: str):
        if not isinstance(value, str) or not self.EMAIL_PATTERN.match(value.strip()):
            raise InvalidCandidateError(f"Invalid email address: {value!r}")
        self.__email = value.strip()

    @property
    def qualification(self) -> str:
        return self.__qualification

    @qualification.setter
    def qualification(self, value: str):
        if not isinstance(value, str) or not value.strip():
            raise InvalidCandidateError("Qualification cannot be empty.")
        self.__qualification = value.strip()

    @property
    def skills(self) -> List[str]:
        return list(self.__skills)

    @skills.setter
    def skills(self, value):
        self.__skills = clean_skills(value, InvalidCandidateError)

    @property
    def role(self) -> JobRole:
        return self.__role

    @role.setter
    def role(self, value):
        if not isinstance(value, JobRole):
            raise InvalidCandidateError("Role must be a JobRole object (e.g. MLEngineer()).")
        self.__role = value

    @abstractmethod
    def get_extra_details(self) -> Dict[str, str]:
        pass

    def match_skills(self, job: "Job") -> Tuple[List[str], List[str]]:
        have = {s.lower() for s in self.skills}
        matched = [s for s in job.required_skills if s.lower() in have]
        missing = [s for s in job.required_skills if s.lower() not in have]
        return matched, missing

    def check_qualification(self, job: "Job") -> bool:
        return job.accepts_qualification(self.qualification)

    def match_keywords(self, keywords: List[str]) -> Tuple[int, List[str]]:
        score, matched = 0, []
        for kw in keywords:
            terms = expand(kw)
            for skill in self.skills:
                if any(t in skill.lower() for t in terms):
                    if skill not in matched:
                        matched.append(skill)
                    score += 2
            if any(t in self.role.title.lower() for t in terms):
                score += 1
            if any(t in self.role.category.lower() for t in terms):
                score += 1
            if any(t in self.name.lower() for t in terms):
                score += 1
        return score, matched

    def get_search_keywords(self) -> List[str]:
        return self.skills + [self.role.title, self.role.category]

    def get_search_lines(self, matched, numbers) -> List[str]:
        skills = ", ".join(
            f"[{s} #{numbers[s.lower()][1]}]" if s in matched else s for s in self.skills
        )
        return [f"{self.name} - {self.role.title} ({self.role.category}) "
                f"[{self.get_candidate_type()}]",
                f"   Work:   {self.role.work()}",
                f"   Skills: {skills}"]

    def get_name(self) -> str:
        return self.name

    def get_summary(self) -> str:
        return f"{self.name} ({self.get_candidate_type()}, {self.role.title})"

    def display_details(self) -> None:
        print(f"  ID            : {self.candidate_id}")
        print(f"  Name          : {self.name}")
        print(f"  Type          : {self.get_candidate_type()}")
        print(f"  Role          : {self.role.title} ({self.role.category})")
        print(f"  Role work     : {self.role.work()}")
        print(f"  Email         : {self.email}")
        print(f"  Qualification : {self.qualification}")
        print(f"  Skills        : {', '.join(self.skills)}")
        for label, value in self.get_extra_details().items():
            print(f"  {label:<14}: {value}")


class FresherCandidate(Candidate):
    def __init__(self, candidate_id, name, email, qualification, skills, role,
                 cgpa: float, internship_months: int = 0):
        super().__init__(candidate_id, name, email, qualification, skills, role)
        self.cgpa = cgpa
        self.internship_months = internship_months

    @property
    def cgpa(self) -> float:
        return self.__cgpa

    @cgpa.setter
    def cgpa(self, value):
        if not isinstance(value, (int, float)) or not 0 <= value <= 10:
            raise InvalidCandidateError("CGPA must be a number between 0 and 10.")
        self.__cgpa = float(value)

    @property
    def internship_months(self) -> int:
        return self.__internship_months

    @internship_months.setter
    def internship_months(self, value):
        if not isinstance(value, int) or value < 0:
            raise InvalidCandidateError("Internship months must be a non-negative integer.")
        self.__internship_months = value

    def get_candidate_type(self) -> str:
        return "Fresher"

    def get_extra_details(self) -> Dict[str, str]:
        return {"CGPA": f"{self.cgpa:.1f}",
                "Internship": f"{self.internship_months} month(s)"}

    def check_experience_eligibility(self, job: "Job") -> Tuple[bool, str]:
        if self.cgpa < job.min_cgpa:
            return False, f"CGPA {self.cgpa:.1f} is below the required {job.min_cgpa:.1f}"
        internship_years = self.internship_months / 12
        if job.min_experience == 0:
            return True, "Entry-level role and CGPA requirement met"
        if internship_years >= job.min_experience:
            return True, (f"Internship experience ({internship_years:.1f} yr) "
                          f"meets the {job.min_experience:g} yr requirement")
        return False, (f"Needs {job.min_experience:g} yr experience, "
                       f"has only {internship_years:.1f} yr of internships")


class ExperiencedCandidate(Candidate):
    def __init__(self, candidate_id, name, email, qualification, skills, role,
                 years_of_experience: float, current_company: str = "N/A"):
        super().__init__(candidate_id, name, email, qualification, skills, role)
        self.years_of_experience = years_of_experience
        self.current_company = current_company

    @property
    def years_of_experience(self) -> float:
        return self.__years_of_experience

    @years_of_experience.setter
    def years_of_experience(self, value):
        if not isinstance(value, (int, float)) or value < 0:
            raise InvalidCandidateError("Years of experience must be a non-negative number.")
        self.__years_of_experience = float(value)

    @property
    def current_company(self) -> str:
        return self.__current_company

    @current_company.setter
    def current_company(self, value):
        self.__current_company = str(value).strip() or "N/A"

    def get_candidate_type(self) -> str:
        return "Experienced"

    def get_extra_details(self) -> Dict[str, str]:
        return {"Experience": f"{self.years_of_experience:g} year(s)",
                "Company": self.current_company}

    def check_experience_eligibility(self, job: "Job") -> Tuple[bool, str]:
        if self.years_of_experience >= job.min_experience:
            return True, (f"{self.years_of_experience:g} yr meets the "
                          f"{job.min_experience:g} yr requirement")
        return False, (f"Needs {job.min_experience:g} yr experience, "
                       f"has only {self.years_of_experience:g} yr")


class Job(Displayable):
    def __init__(self, title: str, company: str, required_skills: List[str],
                 allowed_qualifications: List[str], min_experience: float = 0,
                 min_cgpa: float = 6.0, min_skill_match: float = 60.0):
        if not isinstance(title, str) or not title.strip():
            raise InvalidJobError("Job title cannot be empty.")
        if not isinstance(min_experience, (int, float)) or min_experience < 0:
            raise InvalidJobError("Minimum experience must be non-negative.")
        if not isinstance(min_cgpa, (int, float)) or not 0 <= min_cgpa <= 10:
            raise InvalidJobError("Minimum CGPA must be between 0 and 10.")
        if not isinstance(min_skill_match, (int, float)) or not 0 <= min_skill_match <= 100:
            raise InvalidJobError("Minimum skill match must be between 0 and 100.")

        self.__title = title.strip()
        self.__company = str(company).strip()
        self.__required_skills = clean_skills(required_skills, InvalidJobError)
        self.__allowed_qualifications = clean_skills(allowed_qualifications, InvalidJobError)
        self.__min_experience = float(min_experience)
        self.__min_cgpa = float(min_cgpa)
        self.__min_skill_match = float(min_skill_match)

    @property
    def title(self) -> str:
        return self.__title

    @property
    def company(self) -> str:
        return self.__company

    @property
    def required_skills(self) -> List[str]:
        return list(self.__required_skills)

    @property
    def allowed_qualifications(self) -> List[str]:
        return list(self.__allowed_qualifications)

    @property
    def min_experience(self) -> float:
        return self.__min_experience

    @property
    def min_cgpa(self) -> float:
        return self.__min_cgpa

    @property
    def min_skill_match(self) -> float:
        return self.__min_skill_match

    def accepts_qualification(self, qualification: str) -> bool:
        wanted = {normalize(q) for q in self.allowed_qualifications}
        return normalize(qualification) in wanted

    def get_summary(self) -> str:
        return f"{self.title} ({self.company})"

    def display_details(self) -> None:
        print(f"  Position            : {self.get_summary()}")
        print(f"  Required skills     : {', '.join(self.required_skills)}")
        print(f"  Qualifications      : {', '.join(self.allowed_qualifications)}")
        print(f"  Min. experience     : {self.min_experience:g} year(s)")
        print(f"  Min. CGPA (fresher) : {self.min_cgpa:.1f}")
        print(f"  Min. skill match    : {self.min_skill_match:.0f}%")


@dataclass(frozen=True)
class ScreeningResult:
    candidate: Screenable
    skill_percentage: float
    matched_skills: List[str]
    missing_skills: List[str]
    experience_eligible: bool
    experience_note: str
    qualification_ok: bool
    status: str
    reasons: List[str] = field(default_factory=list)


class TextTable:
    @staticmethod
    def render(headers: List[str], rows: List[list], aligns: List[str] = None,
               max_widths: List[int] = None) -> List[str]:
        count = len(headers)
        aligns = aligns or ["<"] * count
        max_widths = max_widths or [None] * count
        widths = []
        for i, header in enumerate(headers):
            width = max([len(header)] + [len(str(row[i])) for row in rows])
            widths.append(min(width, max_widths[i]) if max_widths[i] else width)

        def make_lines(cells) -> List[str]:
            wrapped = [textwrap.wrap(str(cell), widths[i]) or [""]
                       for i, cell in enumerate(cells)]
            lines = []
            for k in range(max(len(w) for w in wrapped)):
                parts = [f"{(w[k] if k < len(w) else ''):{aligns[i]}{widths[i]}}"
                         for i, w in enumerate(wrapped)]
                lines.append("  ".join(parts).rstrip())
            return lines

        lines = make_lines(headers)
        lines.append("  ".join("-" * w for w in widths))
        for row in rows:
            lines.extend(make_lines(row))
        return lines


class ResumeScreening:
    SHORTLISTED = "Shortlisted"
    REJECTED = "Rejected"

    def __init__(self, job: Job):
        if not isinstance(job, Job):
            raise InvalidJobError("ResumeScreening needs a valid Job object.")
        self.__job = job
        self.__results: List[ScreeningResult] = []

    @property
    def job(self) -> Job:
        return self.__job

    @property
    def results(self) -> List[ScreeningResult]:
        return list(self.__results)

    def calculate_skill_match(self, candidate: Screenable) -> Tuple[float, List[str], List[str]]:
        matched, missing = candidate.match_skills(self.job)
        percentage = (len(matched) / len(self.job.required_skills)) * 100
        return percentage, matched, missing

    def screen_candidate(self, candidate: Screenable) -> ScreeningResult:
        if not isinstance(candidate, Screenable):
            raise InvalidCandidateError("Only objects implementing Screenable can be screened.")

        percentage, matched, missing = self.calculate_skill_match(candidate)
        eligible, note = candidate.check_experience_eligibility(self.job)
        qualification_ok = candidate.check_qualification(self.job)

        reasons = []
        if percentage < self.job.min_skill_match:
            reasons.append(f"Skill match {percentage:.0f}% is below "
                           f"{self.job.min_skill_match:.0f}%")
        if not eligible:
            reasons.append(note)
        if not qualification_ok:
            reasons.append("Qualification not accepted for this job")

        status = self.REJECTED if reasons else self.SHORTLISTED
        result = ScreeningResult(candidate, percentage, matched, missing,
                                 eligible, note, qualification_ok, status, reasons)
        self.__results.append(result)
        return result

    def screen_all(self, candidates: List[Screenable]) -> None:
        for candidate in candidates:
            try:
                self.screen_candidate(candidate)
            except ResumeScreeningError as err:
                print(f"[Error] Could not screen a candidate: {err}")

    def get_shortlisted(self) -> List[ScreeningResult]:
        return [r for r in self.results if r.status == self.SHORTLISTED]

    def get_rejected(self) -> List[ScreeningResult]:
        return [r for r in self.results if r.status == self.REJECTED]

    def display_report(self) -> None:
        line = "=" * 66
        print(f"\n{line}\n SCREENING FOR: {self.job.title.upper()}\n{line}")
        print("Job requirements")
        self.job.display_details()

        print("\nScreening details\n")
        rows = [[number, res.candidate.get_name(), res.candidate.get_candidate_type(),
                 f"{res.skill_percentage:.0f}%",
                 "Eligible" if res.experience_eligible else "Not eligible",
                 "Accepted" if res.qualification_ok else "Not accepted",
                 res.status.upper()]
                for number, res in enumerate(self.results, start=1)]
        print("\n".join(TextTable.render(
            ["#", "Candidate", "Type", "Skill %", "Experience", "Qualification", "Status"],
            rows, aligns=[">", "<", "<", ">", "<", "<", "<"],
            max_widths=[None, 18, None, None, None, None, None])))

        print("\nSkill match breakdown\n")
        rows = [[number, res.candidate.get_name(),
                 ", ".join(res.matched_skills) or "None",
                 ", ".join(res.missing_skills) or "None"]
                for number, res in enumerate(self.results, start=1)]
        print("\n".join(TextTable.render(
            ["#", "Candidate", "Matched skills", "Missing skills"],
            rows, aligns=[">", "<", "<", "<"], max_widths=[None, 18, 30, 30])))

        shortlisted, rejected = self.get_shortlisted(), self.get_rejected()
        print(f"\nSummary for {self.job.title}")
        print(f"Shortlisted ({len(shortlisted)}):")
        for res in shortlisted:
            print(f"  + {res.candidate.get_summary()} - {res.skill_percentage:.0f}% skill match")
        print(f"Rejected ({len(rejected)}):")
        for res in rejected:
            print(f"  - {res.candidate.get_summary()} - {'; '.join(res.reasons)}")


class CandidateSearch:
    def __init__(self, candidates: List[Searchable]):
        if not candidates or not all(isinstance(c, Searchable) for c in candidates):
            raise InvalidCandidateError(
                "CandidateSearch needs a non-empty list of Searchable objects.")
        self.__candidates = list(candidates)
        self.__keyword_numbers = self.__build_keyword_numbers()

    def __build_keyword_numbers(self) -> Dict[str, Tuple[str, int]]:
        words = {}
        for c in self.__candidates:
            for w in c.get_search_keywords():
                words[w.lower()] = w
        names = sorted(words)
        numbers = random.Random(42).sample(range(1000, 10000), len(names))
        return {n: (words[n], num) for n, num in zip(names, numbers)}

    @staticmethod
    def parse_keywords(query: str) -> List[str]:
        return [k.strip().lower() for k in query.split(",") if k.strip()]

    def search(self, query: str) -> List[Tuple[int, Searchable, List[str]]]:
        keywords = self.parse_keywords(query)
        if not keywords:
            raise ResumeScreeningError("Please enter at least one keyword.")
        results = []
        for cand in self.__candidates:
            score, matched = cand.match_keywords(keywords)
            if score > 0:
                results.append((score, cand, matched))
        results.sort(key=lambda r: (-r[0], r[1].get_summary()))
        return results

    def show_results(self, results, query: str) -> None:
        for kw in self.parse_keywords(query):
            if kw in self.__keyword_numbers:
                name, number = self.__keyword_numbers[kw]
                print(f"Keyword: {name}  ->  Number: {number}")

        if not results:
            print(f"No candidates found for '{query}'.")
            return
        print(f"Found {len(results)} result(s) for '{query}':\n")

        for i, (score, cand, matched) in enumerate(results, 1):
            first, *rest = cand.get_search_lines(matched, self.__keyword_numbers)
            print(f"{i}. {first}")
            for line in rest:
                print(line)

    def show_keyword_numbers(self) -> None:
        for name, number in sorted(self.__keyword_numbers.values()):
            print(f"{number}  {name}")

    def run(self) -> None:
        print("\nSearch AI/ML and Gen AI candidates by name, role or skill "
              "(e.g. GenAI, AI/ML, PyTorch).")
        print("Use commas for several keywords. Type 'list' to see all keyword numbers, 'exit' to go back.")
        while True:
            try:
                query = input("\n" + InputHelper.field("Search keywords (or 'exit')")).strip()
            except (EOFError, KeyboardInterrupt):
                break
            if query.lower() == "exit":
                break
            if not query:
                continue
            print()
            if query.lower() == "list":
                self.show_keyword_numbers()
                continue
            try:
                self.show_results(self.search(query), query)
            except ResumeScreeningError as err:
                print(f"[Search Error] {err}")


class InputClosed(Exception):
    pass


class InputHelper:
    LABEL_WIDTH = 42

    @staticmethod
    def field(label: str) -> str:
        return f"  {label:<{InputHelper.LABEL_WIDTH}}: "

    @staticmethod
    def text(label: str, default=None) -> str:
        while True:
            try:
                value = input(InputHelper.field(label)).strip()
            except EOFError:
                raise InputClosed() from None
            if value:
                return value
            if default is not None:
                return str(default)
            print("  This field cannot be empty.")

    @staticmethod
    def validated(label: str, parser, default=None):
        while True:
            raw = InputHelper.text(label, default)
            try:
                return parser(raw)
            except (ResumeScreeningError, ValueError) as err:
                print(f"  Invalid input: {err}")

    @staticmethod
    def number(label: str, low=0, high=None, cast=float, default=None):
        def parse(raw):
            try:
                value = cast(raw)
            except ValueError:
                kind = "whole number" if cast is int else "number"
                raise ValueError(f"'{raw}' is not a valid {kind}.") from None
            if value < low or (high is not None and value > high):
                limit = f"at least {low}" if high is None else f"between {low} and {high}"
                raise ValueError(f"value must be {limit}.")
            return value
        return InputHelper.validated(label, parse, default)

    @staticmethod
    def percent(label: str, default=None) -> float:
        def parse(raw):
            if not raw.endswith("%"):
                raise ValueError("please type the number with the % symbol, e.g. 60%")
            try:
                value = float(raw[:-1].strip())
            except ValueError:
                raise ValueError(f"'{raw}' is not a valid percentage, e.g. 60%") from None
            if not 0 <= value <= 100:
                raise ValueError("the percentage must be between 0% and 100%.")
            return value
        return InputHelper.validated(label, parse, default)

    @staticmethod
    def choice(label: str, valid: List[str]) -> str:
        def parse(raw):
            if raw.lower() not in valid:
                raise ValueError(f"please choose one of: {', '.join(valid)}")
            return raw.lower()
        return InputHelper.validated(label, parse)

    @staticmethod
    def yes_no(label: str, default: bool = True) -> bool:
        def parse(raw):
            answer = raw.lower()
            if answer in ("y", "yes"):
                return True
            if answer in ("n", "no"):
                return False
            raise ValueError("please answer y or n")
        return InputHelper.validated(label, parse, default="y" if default else "n")

    @staticmethod
    def parse_email(raw: str) -> str:
        if not Candidate.EMAIL_PATTERN.match(raw):
            raise InvalidCandidateError("enter a valid email, e.g. name@example.com")
        return raw

    @staticmethod
    def parse_list(raw: str) -> List[str]:
        return clean_skills([x for x in raw.split(",") if x.strip()], ResumeScreeningError)

    @staticmethod
    def parse_role(raw: str) -> JobRole:
        typed = normalize(raw)
        exact = [r for r in ROLE_CLASSES if normalize(r.title) == typed]
        if exact:
            return exact[0]()
        partial = [r for r in ROLE_CLASSES if typed in normalize(r.title)]
        if len(partial) == 1:
            return partial[0]()
        if partial:
            names = ", ".join(r.title for r in partial)
            raise ValueError(f"'{raw}' matches several roles ({names}) - please be more specific.")
        raise ValueError(f"'{raw}' is not a known role - choose one from the list above.")


ROLE_CLASSES = [MLEngineer, MLOpsEngineer, NLPEngineer, ComputerVisionEngineer,
                DataScientist, AIResearcher, GenAIEngineer, PromptEngineer,
                LLMEngineer, AIAgentDeveloper]


def sample_candidate_data() -> List[tuple]:
    return [
        (FresherCandidate, dict(
            name="Arun", email="arun@example.com", qualification="B.Tech",
            role=MLEngineer(),
            skills=["Python", "Scikit-learn", "Machine Learning", "Feature Engineering"],
            cgpa=8.2, internship_months=12)),
        (ExperiencedCandidate, dict(
            name="Bala", email="bala@example.com", qualification="B.E",
            role=MLOpsEngineer(), skills=["Docker", "Kubernetes", "MLflow", "Python"],
            years_of_experience=3, current_company="CloudOps")),
        (FresherCandidate, dict(
            name="Divya", email="divya@example.com", qualification="M.Sc",
            role=PromptEngineer(),
            skills=["Prompt Engineering", "ChatGPT", "Claude", "LLM"],
            cgpa=8.8, internship_months=12)),
        (ExperiencedCandidate, dict(
            name="Kavya", email="kavya@example.com", qualification="MCA",
            role=GenAIEngineer(), skills=["GenAI", "LLM", "LangChain", "RAG", "Python"],
            years_of_experience=2, current_company="PromptWorks")),
        (FresherCandidate, dict(
            name="Meena", email="meena@example.com", qualification="B.Sc",
            role=NLPEngineer(), skills=["NLP", "Transformers", "Hugging Face", "Python"],
            cgpa=7.9, internship_months=8)),
        (ExperiencedCandidate, dict(
            name="Nila", email="nila@example.com", qualification="B.Tech",
            role=ComputerVisionEngineer(),
            skills=["OpenCV", "Deep Learning", "PyTorch", "Python"],
            years_of_experience=2.5, current_company="VisionAI")),
        (ExperiencedCandidate, dict(
            name="Priya", email="priya@example.com", qualification="M.Tech",
            role=DataScientist(), skills=["Statistics", "Machine Learning", "SQL", "Python"],
            years_of_experience=4, current_company="DataMinds")),
        (FresherCandidate, dict(
            name="Ravi", email="ravi@example.com", qualification="B.E",
            role=AIAgentDeveloper(),
            skills=["Agentic AI", "LLM", "LangGraph", "Tool Calling"],
            cgpa=6.0, internship_months=4)),
        (ExperiencedCandidate, dict(
            name="Suresh", email="suresh@example.com", qualification="B.Tech",
            role=LLMEngineer(),
            skills=["LLM", "Fine-tuning", "GenAI", "Vector Databases", "RAG"],
            years_of_experience=3, current_company="LangLabs")),
        (FresherCandidate, dict(
            name="Vimal", email="vimal@example.com", qualification="M.Tech",
            role=AIResearcher(),
            skills=["Deep Learning", "PyTorch", "Research Papers", "Mathematics"],
            cgpa=9.1, internship_months=0)),
    ]


def sample_jobs() -> List[Job]:
    qualifications = ["B.E", "B.Tech", "M.Tech", "MCA", "M.Sc"]
    try:
        return [
            Job(title="GenAI Engineer", company="TechNova Solutions",
                required_skills=["Python", "LLM", "GenAI", "RAG", "LangChain"],
                allowed_qualifications=qualifications,
                min_experience=1, min_cgpa=7.0, min_skill_match=60),
            Job(title="AI/ML Engineer", company="TechNova Solutions",
                required_skills=["Python", "Machine Learning", "Deep Learning", "PyTorch", "SQL"],
                allowed_qualifications=qualifications,
                min_experience=1, min_cgpa=7.0, min_skill_match=60),
        ]
    except InvalidJobError as err:
        print(f"[Job Error] {err}")
        return []


class ResumeScreeningApp:
    def __init__(self):
        self.__candidates: List[Candidate] = []
        self.__jobs: List[Job] = []
        self.__next_id = 101

    def __register_candidate(self, candidate_class, **details) -> Candidate:
        candidate = candidate_class(candidate_id=f"C{self.__next_id}", **details)
        self.__candidates.append(candidate)
        self.__next_id += 1
        return candidate

    @staticmethod
    def __choose_role() -> JobRole:
        print("  Available roles")
        for category in dict.fromkeys(r.category for r in ROLE_CLASSES):
            heading = f"    {category:<6}: "
            titles = ", ".join(r.title.replace(" ", "\u00a0")
                               for r in ROLE_CLASSES if r.category == category)
            print(textwrap.fill(titles, width=78, initial_indent=heading,
                                subsequent_indent=" " * len(heading)).replace("\u00a0", " "))
        return InputHelper.validated("Role name", InputHelper.parse_role)

    def __ask_job(self) -> None:
        print("\nSTEP 1 of 3 - JOB OPENING")
        print("  Press Enter to accept a value shown in [brackets].\n")
        while True:
            title = InputHelper.text("Job title")
            company = InputHelper.text("Company")
            skills = InputHelper.validated("Required skills (comma-separated)",
                                           InputHelper.parse_list)
            qualifications = InputHelper.validated(
                "Accepted qualifications (comma-separated)", InputHelper.parse_list)
            min_exp = InputHelper.number("Minimum experience in years [0]", 0, default=0)
            min_cgpa = InputHelper.number("Minimum CGPA for freshers, 0-10 [6.0]",
                                          0, 10, default=6.0)
            min_match = InputHelper.percent("Minimum skill match, with % [60%]",
                                            default="60%")
            try:
                job = Job(title, company, skills, qualifications, min_exp, min_cgpa, min_match)
            except InvalidJobError as err:
                print(f"[Error] {err} Please enter the job again.\n")
                continue
            self.__jobs.append(job)
            print(f"\n  Job saved: {job.get_summary()}")
            return

    def __ask_candidate(self) -> None:
        kind = InputHelper.choice("Candidate type (1 Fresher, 2 Experienced)", ["1", "2"])
        name = InputHelper.text("Name")
        email = InputHelper.validated("Email", InputHelper.parse_email)
        qualification = InputHelper.text("Qualification (e.g. B.Tech, MCA)")
        skills = InputHelper.validated("Skills (comma-separated)", InputHelper.parse_list)
        role = self.__choose_role()
        common = dict(name=name, email=email, qualification=qualification,
                      skills=skills, role=role)
        try:
            if kind == "1":
                cgpa = InputHelper.number("CGPA (0-10)", 0, 10)
                months = InputHelper.number("Internship months [0]",
                                            0, cast=int, default=0)
                candidate = self.__register_candidate(FresherCandidate, cgpa=cgpa,
                                                      internship_months=months, **common)
            else:
                years = InputHelper.number("Years of experience", 0)
                company = InputHelper.text("Current company [N/A]", default="N/A")
                candidate = self.__register_candidate(ExperiencedCandidate,
                                                      years_of_experience=years,
                                                      current_company=company, **common)
        except InvalidCandidateError as err:
            print(f"[Error] Candidate not added: {err}")
            return
        print(f"\n  Candidate saved: {candidate.get_summary()} (ID {candidate.candidate_id})")

    def __ask_candidates(self) -> None:
        print("\nSTEP 2 of 3 - CANDIDATES")
        print("  Press Enter to accept a value shown in [brackets].")
        while True:
            print(f"\n--- Candidate {len(self.__candidates) + 1} ---")
            self.__ask_candidate()
            if not self.__candidates:
                print("At least one candidate is needed - please try again.")
                continue
            print()
            if not InputHelper.yes_no("Add another candidate? [Y/n]"):
                return

    def __load_sample_data(self) -> None:
        for candidate_class, details in sample_candidate_data():
            try:
                self.__register_candidate(candidate_class, **details)
            except InvalidCandidateError as err:
                print(f"[Skipped] {details['name']}: {err}")
        self.__jobs.extend(sample_jobs())
        print(f"  Loaded {len(self.__candidates)} sample candidates and "
              f"{len(self.__jobs)} job openings.")

    def __show_results(self) -> None:
        line = "=" * 66
        print(f"\nSTEP 3 of 3 - RESULTS\n\n{line}\n CANDIDATE DETAILS\n{line}")
        for number, candidate in enumerate(self.__candidates, start=1):
            print(f"\nCandidate {number}")
            candidate.display_details()
        for job in self.__jobs:
            screening = ResumeScreening(job)
            screening.screen_all(self.__candidates)
            screening.display_report()

    def __offer_search(self) -> None:
        print()
        if InputHelper.yes_no("Search candidates by keyword? [y/N]", default=False):
            CandidateSearch(self.__candidates).run()

    def run(self) -> None:
        print("=" * 50)
        print(" RESUME SCREENING SYSTEM")
        print("=" * 50)
        print()
        if InputHelper.yes_no("Use built-in sample data? [y/N]", default=False):
            self.__load_sample_data()
        else:
            self.__ask_job()
            self.__ask_candidates()
        self.__show_results()
        self.__offer_search()
        print("\nDone. Thank you!")


def main() -> None:
    try:
        ResumeScreeningApp().run()
    except (InputClosed, KeyboardInterrupt):
        print("\nGoodbye!")


if __name__ == "__main__":
    main()

 RESUME SCREENING SYSTEM



  Use built-in sample data? [y/N]           :  N



STEP 1 of 3 - JOB OPENING
  Press Enter to accept a value shown in [brackets].



  Job title                                 :  GenAI
  Company                                   :  TechNova Solutions
  Required skills (comma-separated)         :  Python, LLM, GenAI, RAG, LangChain
  Accepted qualifications (comma-separated) :   B.E, B.Tech, M.Tech, MCA, M.Sc
  Minimum experience in years [0]           :  1
  Minimum CGPA for freshers, 0-10 [6.0]     :  6.0
  Minimum skill match, with % [60%]         :  60%



  Job saved: GenAI (TechNova Solutions)

STEP 2 of 3 - CANDIDATES
  Press Enter to accept a value shown in [brackets].

--- Candidate 1 ---


  Candidate type (1 Fresher, 2 Experienced) :  2
  Name                                      :  Kavya
  Email                                     :  kavya@gmail.com
  Qualification (e.g. B.Tech, MCA)          :  MCA
  Skills (comma-separated)                  :  Python, LLM, GenAI, RAG, LangChain


... (19 outputs left out)


  Search keywords (or 'exit')               :  list



1409  AI Agent Developer
2824  Agentic AI
5506  Gen AI
5012  GenAI
4657  GenAI Engineer
9935  LLM
3286  LangChain
2679  LangGraph
2424  Python
7912  RAG
1520  Tool Calling



  Search keywords (or 'exit')               :  Exit



Done. Thank you!
